In [1]:
import pandas as pd
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments

In [2]:
# Load Dataset

train_data = pd.read_csv("/content/data/samsum-train.csv")
validation_data = pd.read_csv("/content/data/samsum-validation.csv")

print(train_data.head())
print(f"Train Shape: {train_data.shape}")
print(f"Validation Shape: {validation_data.shape}")

         id                                           dialogue  \
0  13818513  Amanda: I baked  cookies. Do you want some?\r\...   
1  13728867  Olivia: Who are you voting for in this electio...   
2  13681000  Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...   
3  13730747  Edward: Rachel, I think I'm in ove with Bella....   
4  13728094  Sam: hey  overheard rick say something\r\nSam:...   

                                             summary  
0  Amanda baked cookies and will bring Jerry some...  
1  Olivia and Olivier are voting for liberals in ...  
2  Kim may try the pomodoro technique recommended...  
3  Edward thinks he is in love with Bella. Rachel...  
4  Sam is confused, because he overheard Rick com...  
Train Shape: (14732, 3)
Validation Shape: (818, 3)


In [3]:
# Select random samples for Train and Validation dataset

train_data = train_data.sample(n = 5000, random_state = 42).reset_index(drop = True)
validation_data = validation_data.sample(n = 500, random_state = 42).reset_index(drop = True)

print(f"Train Shape: {train_data.shape}")
print(f"Validation Shape: {validation_data.shape}")

Train Shape: (5000, 3)
Validation Shape: (500, 3)


In [4]:
# Clean Text

import re

def clean_text(text):
  text = re.sub(r"\r\n", " ", text)
  text = re.sub(r"\s+", " ", text)
  text = re.sub(r"<.*?>", " ", text)
  text = text.strip().lower()

  return text

train_data["dialogue"] = train_data["dialogue"].apply(clean_text)
train_data["summary"] = train_data["summary"].apply(clean_text)

validation_data["dialogue"] = validation_data["dialogue"].apply(clean_text)
validation_data["summary"] = validation_data["summary"].apply(clean_text)

In [5]:
# Load Tokenzer

tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
# Apply Tokenization

def preprocess_function(examples):
  inputs = tokenizer(examples["dialogue"], max_length = 512, padding = "max_length", truncation = True)
  targets = tokenizer(examples["summary"], max_length = 175, padding = "max_length", truncation = True)

  inputs["labels"] = targets["input_ids"]

  return inputs

train_data = train_data.apply(preprocess_function, axis = 1)
validation_data = validation_data.apply(preprocess_function, axis = 1)

In [7]:
# Set Model

model = T5ForConditionalGeneration.from_pretrained("t5-small")

training_args = TrainingArguments(
    num_train_epochs = 6,
    output_dir = "./results",
    logging_dir = "./logs",
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    warmup_steps = 500,
    weight_decay = 0.01,
    save_steps = 500,
    eval_steps = 50,
    do_eval = True
)

trainer = Trainer(
 model = model,
 args = training_args,
 train_dataset = train_data,
 eval_dataset = validation_data
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [8]:
# Train Model

trainer.train()

Step,Training Loss
500,3.599096
1000,0.345293
1500,0.325102
2000,0.315499
2500,0.309939
3000,0.304841
3500,0.301344


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=0.7534383911132813, metrics={'train_runtime': 1603.8068, 'train_samples_per_second': 18.705, 'train_steps_per_second': 2.338, 'total_flos': 4060254044160000.0, 'train_loss': 0.7534383911132813, 'epoch': 6.0})

In [20]:
# Save Models

model.save_pretrained("./saved_files/summarization_model")
tokenizer.save_pretrained("./saved_files/tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_files/tokenizer/tokenizer_config.json',
 './saved_files/tokenizer/tokenizer.json')

In [21]:
# Use Models

model = T5ForConditionalGeneration.from_pretrained("./saved_files/summarization_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_files/tokenizer")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [23]:
# Build Summarization Model

device = model.device

def summarize_dialogue(dialogue):
  dialogue = clean_text(dialogue)
  # tokenizer return: input_ids, attention_mask
  # tokenizer.encode return: input_ids
  input_ids = tokenizer(dialogue, return_tensors = "pt", max_length = 512, truncation = True)

  inputs = {key: value.to(device) for key, value in input_ids.items()}

  outputs = model.generate(
    input_ids["input_ids"],
    max_length = 175,
    num_beams = 4,
    early_stopping = True,
  )

  summary = tokenizer.decode(outputs[0], skip_special_tokens = True)
  return summary

In [24]:
# Test Sample

origianl_dialogue = """
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)",Eric and Rob are going to watch a stand-up on youtube.
"""

summary = summarize_dialogue(origianl_dialogue)
print(f"Summary: {summary}")

Summary: eric and rob are going to watch a stand-up on youtube.


In [27]:
# Archive Model

import shutil

shutil.make_archive("summarization_model", "zip", "./saved_files")

from google.colab import files

files.download("summarization_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>